In [0]:
%sql
-- ==========================================================================
-- POPULATE DIMENSION AND FACT TABLES FROM CLEAN LAYER
-- ==========================================================================

-- ----------------------------------------------------------
-- DIM: DimCourse
-- Grain: one row per course module
-- ----------------------------------------------------------
INSERT OVERWRITE DimCourse
SELECT DISTINCT
    code_module
FROM oulad.clean.courses;

-- ----------------------------------------------------------
-- DIM: DimDate
-- Grain: one row per unique date
-- Generate from all date columns across tables
-- ----------------------------------------------------------
INSERT OVERWRITE DimDate
WITH all_dates AS (
    -- Dates from assessments
    SELECT DISTINCT date AS date_value
    FROM oulad.clean.01_clean_assessments
    WHERE date IS NOT NULL
    
    UNION
    
    -- Dates from student_vle
    SELECT DISTINCT date AS date_value
    FROM oulad.clean.student_vle
    WHERE date IS NOT NULL
    
    UNION
    
    -- Registration dates
    SELECT DISTINCT date_registration AS date_value
    FROM oulad.clean.student_registration
    WHERE date_registration IS NOT NULL
    
    UNION
    
    -- Unregistration dates
    SELECT DISTINCT date_unregistration AS date_value
    FROM oulad.clean.student_registration
    WHERE date_unregistration IS NOT NULL
    
    UNION
    
    -- Submission dates
    SELECT DISTINCT date_submitted AS date_value
    FROM oulad.clean.student_assessment
    WHERE date_submitted IS NOT NULL
)
SELECT
    date_value AS date,
    FLOOR(date_value / 7) AS relative_week,
    CASE
        WHEN date_value < 0 THEN 'Pre-course'
        WHEN date_value BETWEEN 0 AND 90 THEN 'Early'
        WHEN date_value BETWEEN 91 AND 180 THEN 'Mid'
        WHEN date_value > 180 THEN 'Late'
        ELSE 'Unknown'
    END AS course_phase
FROM all_dates;

-- ----------------------------------------------------------
-- DIM: DimStudent
-- Grain: one row per student
-- Combines student_info and student_registration
-- ----------------------------------------------------------
INSERT OVERWRITE DimStudent
SELECT DISTINCT
    si.id_student,
    si.final_result,
    sr.date_registration,
    sr.date_unregistration,
    si.is_withdrawn
FROM oulad.clean.student_info si
LEFT JOIN oulad.clean.student_registration sr
    ON si.id_student = sr.id_student
    AND si.code_module = sr.code_module
    AND si.code_presentation = sr.code_presentation
WHERE si.id_student IS NOT NULL;

-- ----------------------------------------------------------
-- DIM: DimModulePresentation
-- Grain: one row per module + presentation combination
-- ----------------------------------------------------------
INSERT OVERWRITE DimModulePresentation
SELECT DISTINCT
    code_module,
    code_presentation,
    module_presentation_length
FROM oulad.clean.courses;

-- ----------------------------------------------------------
-- DIM: DimDemographics
-- Grain: one row per student per module presentation
-- ----------------------------------------------------------
INSERT OVERWRITE DimDemographics
SELECT
    id_student,
    code_module,
    code_presentation,
    gender,
    region,
    highest_education,
    imd_band,
    age_band,
    num_of_prev_attempts AS num_of_previous_attempts,
    studied_credits,
    disability
FROM oulad.clean.student_info
WHERE id_student IS NOT NULL
  AND code_module IS NOT NULL
  AND code_presentation IS NOT NULL;

-- ----------------------------------------------------------
-- FACT: FactVLEInteractions
-- Grain: one row per student per date per site per module presentation
-- ----------------------------------------------------------
INSERT OVERWRITE FactVLEInteractions
SELECT
    sv.id_student,
    sv.code_module,
    sv.code_presentation,
    sv.date,
    sv.id_site,
    v.activity_type,
    sv.sum_click
FROM oulad.clean.student_vle sv
INNER JOIN oulad.clean.vle v
    ON sv.id_site = v.id_site
    AND sv.code_module = v.code_module
    AND sv.code_presentation = v.code_presentation
WHERE sv.id_student IS NOT NULL
  AND sv.date IS NOT NULL
  AND sv.id_site IS NOT NULL;

-- ----------------------------------------------------------
-- FACT: FactAssessments
-- Grain: one row per student per assessment per module presentation
-- ----------------------------------------------------------
INSERT OVERWRITE FactAssessments
SELECT
    sa.id_student,
    a.code_module,
    a.code_presentation,
    a.date,
    sa.id_assessment,
    a.assessment_type,
    a.weight,
    sa.score,
    sa.date_submitted,
    (sa.date_submitted - a.date) AS submission_delay,
    CAST(sa.is_banked AS BOOLEAN) AS is_banked
FROM oulad.clean.student_assessment sa
INNER JOIN oulad.clean.01_clean_assessments a
    ON sa.id_assessment = a.id_assessment
WHERE sa.id_student IS NOT NULL
  AND sa.id_assessment IS NOT NULL
  AND a.date IS NOT NULL;

-- Verification: Show row counts for all tables
SELECT 'DimCourse' AS table_name, COUNT(*) AS row_count FROM DimCourse
UNION ALL
SELECT 'DimDate', COUNT(*) FROM DimDate
UNION ALL
SELECT 'DimStudent', COUNT(*) FROM DimStudent
UNION ALL
SELECT 'DimModulePresentation', COUNT(*) FROM DimModulePresentation
UNION ALL
SELECT 'DimDemographics', COUNT(*) FROM DimDemographics
UNION ALL
SELECT 'FactVLEInteractions', COUNT(*) FROM FactVLEInteractions
UNION ALL
SELECT 'FactAssessments', COUNT(*) FROM FactAssessments
ORDER BY table_name;